In [ ]:
# LaMa on a production-like stream of crops whose shapes vary (moving marks, borrow leftovers):
# per-variant total time incl. TorchScript re-profiling/compiles on new shapes. Fresh model per variant.
import hashlib, os, sys, time, warnings, subprocess, json
import numpy as np, torch
URL = 'https://github.com/Sanster/models/releases/download/add_big_lama/big-lama.pt'
P = '/kaggle/working/big-lama.pt'
if not os.path.exists(P):
    torch.hub.download_url_to_file(URL, P, progress=False)
assert hashlib.md5(open(P, 'rb').read()).hexdigest() == 'e3aa4aaa15225a33ec84f9f4bc47e500'
CHILD = r'''
import sys, time, warnings, json, numpy as np, torch
variant = sys.argv[1]
P = '/kaggle/working/big-lama.pt'
if 'nofuse' in variant:
    torch._C._jit_set_profiling_executor(False)
    torch._C._jit_set_profiling_mode(False)
if 'static0' in variant:
    torch.jit.set_fusion_strategy([('STATIC', 0), ('DYNAMIC', 0)])
dev = torch.device('cuda:0')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    m = torch.jit.load(P, map_location=dev).eval()
rng = np.random.default_rng(0)
def pow2(n): return 1 << (n - 1).bit_length()
def up(n, k): return -(-n // k) * k
shapes = [(int(rng.integers(17, 26)) * 8, int(rng.integers(38, 58)) * 8) for _ in range(150)]   # 136-200 x 304-456
times, first = [], {}
for h, w in shapes:
    if 'fp16' in variant: H, W = pow2(h), pow2(w)
    elif 'pad64' in variant: H, W = up(h, 64), up(w, 64)
    else: H, W = up(h, 8), up(w, 8)
    x = torch.rand(1, 3, H, W, device=dev); y = (torch.rand(1, 1, H, W, device=dev) > 0.8).float()
    torch.cuda.synchronize(); t = time.time()
    with torch.inference_mode():
        if 'fp16' in variant:
            with torch.autocast('cuda', dtype=torch.float16):
                o = m(x, y)
        else:
            o = m(x, y)
        o.float().cpu()
    dt = time.time() - t
    times.append(dt)
print(json.dumps({'variant': variant, 'total': sum(times), 'first5': [round(t * 1000) for t in times[:5]],
                  'median_ms': float(np.median(times) * 1000), 'max_ms': max(times) * 1000,
                  'n_over_300ms': int(sum(t > 0.3 for t in times)), 'distinct_shapes': len(set(shapes))}))
'''
open('/kaggle/working/child.py', 'w').write(CHILD)
R = open('/kaggle/working/results.txt', 'w')
for v in ('fp32_pad64', 'fp32_pad8', 'fp16', 'fp32_pad64_nofuse', 'fp32_pad8_nofuse', 'fp16_nofuse',
          'fp32_pad8_static0', 'fp16_static0'):
    r = subprocess.run([sys.executable, '/kaggle/working/child.py', v], capture_output=True, text=True)
    line = (r.stdout.strip().splitlines() or ['FAILED ' + r.stderr[-400:]])[-1]
    print(line, flush=True); R.write(line + '\n'); R.flush()
